In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
INPUT_FILE = "../data/raw/acndata_sessions.json"
OUTPUT_FILE = "../data/processed/acn_sessions_clean.csv"

In [3]:
rows = []

with open(INPUT_FILE, "r") as f:
    data = json.load(f)

for session in data["_items"]:

    row = {
        "connectionTime": session.get("connectionTime"),
        "disconnectTime": session.get("disconnectTime"),
        "doneChargingTime": session.get("doneChargingTime"),

        "kWhDelivered": session.get("kWhDelivered"),

        "stationID": session.get("stationID"),

        "kWhRequested": None,
        "minutesAvailable": None,
    }

    user_inputs = session.get("userInputs")

    if user_inputs and len(user_inputs) > 0:

        last_input = user_inputs[-1]

        row["kWhRequested"] = last_input.get("kWhRequested")
        row["minutesAvailable"] = last_input.get("minutesAvailable")

    rows.append(row)

df = pd.DataFrame(rows)


In [4]:

datetime_cols = [
    "connectionTime",
    "disconnectTime",
    "doneChargingTime"
]

for col in datetime_cols:
    df[col] = pd.to_datetime(
        df[col],
        utc=True,
        errors="coerce"
    )

for col in datetime_cols:
    df[col] = (
        df[col]
        .dt.tz_convert("America/Los_Angeles")
        .dt.tz_localize(None)
    )
    

df["sessionDurationHours"] = ( df["disconnectTime"] - df["connectionTime"]).dt.total_seconds() / 3600
df["chargingDurationHours"] = ( df["doneChargingTime"] - df["connectionTime"]).dt.total_seconds() / 3600
df["idleDurationHours"] = ( df["disconnectTime"] - df["doneChargingTime"]).dt.total_seconds() / 3600
df = df[ (df["sessionDurationHours"] >= 0) & (df["chargingDurationHours"] >= 0)]
df["idleDurationHours"] = df["idleDurationHours"].clip(lower=0) 



In [5]:

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved:", OUTPUT_FILE)
print("Rows:", len(df))

print("\nMissing Values (%)")
print(
    (df.isna().mean() * 100)
    .sort_values(ascending=False)
)

Saved: ../data/processed/acn_sessions_clean.csv
Rows: 14973

Missing Values (%)
minutesAvailable         85.106525
kWhRequested             85.106525
disconnectTime            0.000000
connectionTime            0.000000
kWhDelivered              0.000000
doneChargingTime          0.000000
stationID                 0.000000
sessionDurationHours      0.000000
chargingDurationHours     0.000000
idleDurationHours         0.000000
dtype: float64
